# NB13 — ElasticNet: Best of Ridge and Lasso

> **StatQuest: "ElasticNet mixes L1 and L2 — you get sparsity like Lasso AND stability like Ridge."**

Minimises: **SSR + alpha * [ l1_ratio * sum(|b|) + (1-l1_ratio) * sum(b^2)/2 ]**

- `l1_ratio = 0` -> Ridge
- `l1_ratio = 1` -> Lasso
- `0 < l1_ratio < 1` -> ElasticNet

ElasticNet handles correlated features better than Lasso:
Lasso picks ONE from a group; ElasticNet can keep several (with smaller coefficients).


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

def flow_diagram(steps, title, colors=None, notes=None, figsize=(14, 2.8)):
    n = len(steps)
    default_colors = ['#1565C0','#2E7D32','#E65100','#6A1B9A',
                      '#00695C','#AD1457','#37474F','#4E342E',
                      '#0277BD','#558B2F','#C62828','#F57F17']
    colors = (colors or default_colors)[:n]
    notes  = notes or ['']*n
    fig, ax = plt.subplots(figsize=figsize)
    ax.set_xlim(-0.3, n*3.1); ax.set_ylim(-1.2, 2.4); ax.axis('off')
    bw, bh = 2.6, 1.3
    for i,(step,color,note) in enumerate(zip(steps,colors,notes)):
        x = i*3.1
        box = FancyBboxPatch((x,0.2),bw,bh,boxstyle="round,pad=0.12",
                             facecolor=color,edgecolor='white',linewidth=1.5,alpha=0.90)
        ax.add_patch(box)
        ax.text(x+bw/2,0.2+bh/2,step,ha='center',va='center',fontsize=8.5,
                color='white',fontweight='bold',multialignment='center')
        if note:
            ax.text(x+bw/2,0.02,note,ha='center',va='top',fontsize=7,
                    color='#555',style='italic')
        if i < n-1:
            ax.annotate('',xy=(x+bw+0.38,0.2+bh/2),xytext=(x+bw+0.08,0.2+bh/2),
                       arrowprops=dict(arrowstyle='->',color='#444',lw=2.2))
    ax.set_title(title,fontsize=11,fontweight='bold',pad=6,color='#222')
    plt.tight_layout(pad=0.4); plt.show()

flow_diagram(
    steps=[
        'Many correlated\nfeatures\n(Lasso unstable)',
        'ElasticNet:\nL1 + L2\npenalty mix',
        'l1_ratio\ncontrols\nL1/L2 balance',
        'alpha controls\noverall\nshrinkage',
        'Tune both\nwith\nElasticNetCV',
        'Result:\nSparse + stable\ncoefficients',
    ],
    title='NB13 Conceptual Flow: ElasticNet (L1 + L2 Combined)',
    colors=['#C62828','#1565C0','#2E7D32','#E65100','#6A1B9A','#00695C'],
)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge, Lasso, ElasticNet, ElasticNetCV
from sklearn.datasets import make_regression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score

# Dataset with many correlated features
X, y, true_coef = make_regression(n_samples=200, n_features=20,
                                   n_informative=5, noise=10,
                                   coef=True, random_state=0)
X_std = StandardScaler().fit_transform(X)

models = {
    'Ridge (l1=0)':         Ridge(alpha=1),
    'Lasso (l1=1)':         Lasso(alpha=0.1),
    'ElasticNet (l1=0.5)':  ElasticNet(alpha=0.1, l1_ratio=0.5),
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, m) in zip(axes, models.items()):
    m.fit(X_std, y)
    cv_r2 = cross_val_score(m.__class__(**m.get_params()), X_std, y, cv=5).mean()
    nonzero = np.sum(m.coef_ != 0)
    colors = ['#1565C0' if c != 0 else '#BDBDBD' for c in m.coef_]
    ax.bar(range(20), m.coef_, color=colors)
    ax.axhline(0, color='black', linewidth=0.7)
    ax.set_title(f'{name}\n{nonzero}/20 non-zero  CV R^2={cv_r2:.3f}', fontsize=9)
    ax.set_xlabel('Feature index'); ax.grid(alpha=0.3, axis='y')

plt.suptitle('ElasticNet: Lasso-like sparsity + Ridge-like stability on correlated features',
             fontsize=11)
plt.tight_layout(); plt.show()


In [ ]:
# Tune both alpha and l1_ratio simultaneously
from sklearn.linear_model import ElasticNetCV
import numpy as np

en_cv = ElasticNetCV(
    l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 1.0],
    n_alphas=100, cv=5, max_iter=10000
).fit(X_std, y)

print(f"Optimal alpha   : {en_cv.alpha_:.6f}")
print(f"Optimal l1_ratio: {en_cv.l1_ratio_:.2f}")
print(f"Non-zero coefs  : {np.sum(en_cv.coef_ != 0)}/{X.shape[1]}")
print(f"R^2 (train)     : {en_cv.score(X_std, y):.4f}")
print()
print("Rule of thumb: l1_ratio in [0.5, 0.9] works well for most problems")


## When to choose which regulariser

| Situation | Best choice |
|-----------|------------|
| All features probably matter | Ridge |
| Few features out of many matter | Lasso |
| Many correlated features, want some selection | ElasticNet |
| You don't know | Try all three with cross-validation |

**Next: NB14 — Cross-validation and model selection.**
